# Ejemplo 3 — Sintonía PD mediante Root Locus

En este cuaderno se verifica el diseño analítico de un controlador PD para la planta

$$G(s)=\frac{1}{(s+1)(s+2)(s+10)}.$$

El objetivo es conservar un coeficiente de amortiguamiento

$$\zeta=0.5,$$

pero reducir aproximadamente a la mitad el tiempo de establecimiento obtenido con la sintonía proporcional.

Para el controlador proporcional se obtuvo aproximadamente

$$T_s\approx3.25\ \text{s}.$$

Por tanto, para el diseño PD se fija

$$T_s\approx1.625\ \text{s}.$$

A partir de estas especificaciones se obtiene el polo deseado

$$s_d\approx-2.462+4.264j.$$

El controlador tendrá la forma

$$C_{PD}(s)=K(s+z_D).$$

La condición de ángulo se utilizará para verificar la ubicación del cero y la condición de magnitud para calcular $K$.

In [ ]:
!pip -q install control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import control as ct

## Definición de la planta

In [ ]:
s = ct.TransferFunction.s
G = 1 / ((s + 1)*(s + 2)*(s + 10))

G

## Cálculo del polo deseado

Usando la aproximación

$$T_s\approx\frac{4}{\sigma},$$

con $T_s=1.625$ s se obtiene

$$\sigma=\frac{4}{T_s}.$$

Para $\zeta=0.5$,

$$\omega_d=\sigma\frac{\sqrt{1-\zeta^2}}{\zeta}.$$

Así,

$$s_d=-\sigma+j\omega_d.$$

In [ ]:
zeta = 0.5
Ts = 1.625

sigma = 4/Ts
omega_d = sigma*np.sqrt(1-zeta**2)/zeta
sd = -sigma + 1j*omega_d

print(f"sigma   = {sigma:.6f}")
print(f"omega_d = {omega_d:.6f}")
print(f"sd      = {sd.real:.6f} {sd.imag:+.6f}j")

## Condición de ángulo

Los polos de la planta están ubicados en $-1$, $-2$ y $-10$. Primero calculamos los ángulos que forman los vectores desde cada polo hasta $s_d$.

Como el controlador PD introduce un cero en $s=-z_D$, el ángulo que debe aportar este cero es

$$\theta_z=\sum \theta_p-180^\circ.$$

In [ ]:
polos_planta = [-1, -2, -10]

angulos_polos = []

for p in polos_planta:
    angulo = np.degrees(np.angle(sd - p))
    angulos_polos.append(angulo)
    print(f"Ángulo desde el polo {p:>3}: {angulo:.4f}°")

suma_angulos = sum(angulos_polos)
theta_z = suma_angulos - 180

print(f"\nSuma de ángulos de los polos = {suma_angulos:.4f}°")
print(f"Ángulo requerido del cero     = {theta_z:.4f}°")

## Ubicación del cero

Geométricamente,

$$\tan\theta_z=\frac{\omega_d}{z_D-\sigma}.$$

Por tanto,

$$z_D=\sigma+\frac{\omega_d}{\tan\theta_z}.$$

In [ ]:
zD = sigma + omega_d/np.tan(np.deg2rad(theta_z))

print(f"z_D = {zD:.6f}")
print(f"El cero del controlador está en s = {-zD:.6f}")

## Condición de magnitud

Con

$$L(s)=K\frac{s+z_D}{(s+1)(s+2)(s+10)},$$

la condición de magnitud proporciona

$$K=\frac{|s_d+1|\,|s_d+2|\,|s_d+10|}{|s_d+z_D|}.$$

In [ ]:
K = abs((sd + 1)*(sd + 2)*(sd + 10)) / abs(sd + zD)

print(f"K = {K:.6f}")

## Ganancias equivalentes del controlador PD

Al expandir

$$C_{PD}(s)=K(s+z_D)=Ks+Kz_D,$$

y compararlo con

$$C_{PD}(s)=K_Ds+K_P,$$

se obtiene

$$K_D=K,\qquad K_P=Kz_D.$$

In [ ]:
KD = K
KP = K*zD

print(f"K_P = {KP:.6f}")
print(f"K_D = {KD:.6f}")

## Polos de lazo cerrado

Se construye el controlador PD y se verifican los polos del sistema de lazo cerrado.

In [ ]:
C_PD = K*(s + zD)
T_PD = ct.feedback(C_PD*G, 1)

polos_PD = ct.poles(T_PD)

print("Polos del sistema de lazo cerrado:")
for polo in polos_PD:
    print(f"{polo.real:.6f} {polo.imag:+.6f}j")

## Root Locus

Se grafica el Root Locus del sistema compensado sin incluir la ganancia $K$ y se marcan los polos deseados.

In [ ]:
G_PD = (s + zD) / ((s + 1)*(s + 2)*(s + 10))

plt.figure(figsize=(8, 6))
ct.root_locus(G_PD, grid=True)

plt.plot(sd.real, sd.imag, 'rx', markersize=10, markeredgewidth=2,
         label='Polos deseados')
plt.plot(sd.real, -sd.imag, 'rx', markersize=10, markeredgewidth=2)

plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.title('Root Locus con controlador PD')
plt.legend()
plt.grid(True)
plt.show()

## Comparación de la respuesta al escalón

Se compara la respuesta obtenida con el controlador PD con la sintonía proporcional del Ejemplo 1.

In [ ]:
# Controlador proporcional del Ejemplo 1
K_P_ej1 = 43.85
T_P = ct.feedback(K_P_ej1*G, 1)

t = np.linspace(0, 10, 1000)
tP, yP = ct.step_response(T_P, T=t)
tPD, yPD = ct.step_response(T_PD, T=t)

plt.figure(figsize=(8, 5))
plt.plot(tP, yP, label='Control P')
plt.plot(tPD, yPD, label='Control PD')
plt.axhline(1, linestyle='--', linewidth=1, label='Referencia')

plt.xlabel('Tiempo [s]')
plt.ylabel('Salida')
plt.title('Comparación de respuestas al escalón')
plt.grid(True)
plt.legend()
plt.show()

Observe que el cero añadido modifica la dinámica completa del sistema. Aunque los polos deseados pertenecen al Root Locus, conviene comparar la respuesta real con la aproximación de segundo orden y verificar la influencia del tercer polo y del cero.